# Chuyển `customer_turns` thành audio bằng Gemini TTS

Notebook Google Colab này đọc **một file scenario JSON**, lấy `calls.<call_id>.customer_turns`, rồi tạo **một file WAV cho mỗi cuộc gọi** bằng Gemini Text-to-Speech.

Trước khi chạy:

1. Chạy ô cài đặt thư viện và mount Google Drive.
2. Lưu API key với tên `GEMINI_API_KEY` trong **Colab Secrets**, hoặc thêm `GEMINI_API_KEY=...` vào `/content/.env`. Không ghi key trực tiếp vào notebook.
3. Chọn `SCENARIO_FILE`, kiểm tra phần xem trước, rồi đổi `RUN_TTS = True`.

> Mặc định `RUN_TTS = False` để Run All không vô tình sử dụng quota. API tham chiếu: https://ai.google.dev/gemini-api/docs/speech-generation

In [ ]:
%pip install -q -U google-genai python-dotenv

In [ ]:
import base64
import json
import os
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from IPython.display import Audio, display

## 1. Mount Google Drive và chọn file đầu vào

Đổi `SCENARIO_FILE` nếu file của bạn có tên hoặc vị trí khác. Các đường dẫn phải là đối tượng `Path`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content")
DATASET_DIR = Path("/content/drive/MyDrive/scenarios_complete dataset")

# Thay tên file tại đây nếu cần.
SCENARIO_FILE = DATASET_DIR / "SC-001.json"
OUTPUT_DIR = Path("/content")

if not SCENARIO_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {SCENARIO_FILE}")

print("Input :", SCENARIO_FILE)
print("Output:", OUTPUT_DIR)

## 2. Đọc và kiểm tra `customer_turns`

In [ ]:
def load_customer_turns(json_path: Path) -> tuple[str, dict[str, list[str]]]:
    """Đọc scenario và trả về (scenario_id, {call_id: customer_turns})."""
    with json_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError("Nội dung JSON phải là một object.")

    scenario_id = str(payload.get("scenario_id") or json_path.stem)
    calls = payload.get("calls")
    if not isinstance(calls, dict) or not calls:
        raise ValueError("Không tìm thấy object 'calls' hoặc 'calls' đang rỗng.")

    turns_by_call: dict[str, list[str]] = {}
    for call_id, call_data in calls.items():
        if not isinstance(call_data, dict):
            raise ValueError(f"{call_id} phải là một object.")

        turns = call_data.get("customer_turns")
        if not isinstance(turns, list):
            raise ValueError(f"{call_id}.customer_turns phải là một list.")
        if not all(isinstance(turn, str) for turn in turns):
            raise ValueError(f"Mọi phần tử trong {call_id}.customer_turns phải là chuỗi.")

        cleaned_turns = [turn.strip() for turn in turns if turn.strip()]
        if not cleaned_turns:
            raise ValueError(f"{call_id}.customer_turns không có câu hợp lệ.")
        turns_by_call[str(call_id)] = cleaned_turns

    return scenario_id, turns_by_call


scenario_id, customer_turns_by_call = load_customer_turns(SCENARIO_FILE)
print(f"Scenario: {scenario_id}")
for call_id, turns in customer_turns_by_call.items():
    print(f"\n{call_id} ({len(turns)} câu):")
    for index, text in enumerate(turns, start=1):
        print(f"  {index}. {text}")

## 3. Cấu hình Gemini TTS

- `CALL_IDS = None`: xử lý tất cả các cuộc gọi; ví dụ `["call_1"]` để chỉ xử lý một call.
- `Kore` là voice mặc định; có thể thay bằng voice Gemini khác.
- Style được truyền qua `speech_metadata`, không trộn vào nội dung được đọc.
- Các lượt khách được ngăn bằng inline tag `<short pause>`.
- `OVERWRITE = False`: bỏ qua WAV đã tồn tại.

In [ ]:
MODEL = "gemini-3.8-flash-tts"
VOICE = "Kore"
SPEECH_STYLE = (
    "Natural, friendly Vietnamese phone conversation; clear pronunciation; "
    "moderate pace; preserve all numbers, dates, prices, names, and addresses exactly."
)
TURN_SEPARATOR = "\n<short pause>\n"
MAX_RETRIES = 3
RETRY_BASE_SECONDS = 2

CALL_IDS = None  # None = tất cả; ví dụ: ["call_1"]
OVERWRITE = False
RUN_TTS = False  # Đổi thành True sau khi đã kiểm tra nội dung phía trên.

load_dotenv(PROJECT_ROOT / ".env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# Nếu chưa có trong biến môi trường/.env, thử lấy từ Colab Secrets.
if not GEMINI_API_KEY:
    try:
        from google.colab import userdata

        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        GEMINI_API_KEY = None

print("Đã tìm thấy GEMINI_API_KEY:", bool(GEMINI_API_KEY))

In [ ]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "audio"


def build_call_text(text_chunks: list[str]) -> str:
    text = TURN_SEPARATOR.join(chunk.strip() for chunk in text_chunks if chunk.strip())
    if not text:
        raise ValueError("Không có nội dung hợp lệ để tạo audio.")
    return text


def retryable_status_code(exc: Exception) -> int | None:
    for name in ("status_code", "code"):
        value = getattr(exc, name, None)
        try:
            if value is not None:
                return int(value)
        except (TypeError, ValueError):
            pass
    return None


def synthesize_call(
    client: genai.Client,
    scenario_id: str,
    call_id: str,
    text_chunks: list[str],
    output_dir: Path,
) -> tuple[Path, int, dict]:
    """Gọi Gemini TTS và tạo một WAV cho toàn bộ customer_turns của một call."""
    text = build_call_text(text_chunks)
    interaction = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            interaction = client.interactions.create(
                model=MODEL,
                input=[
                    {
                        "type": "user_input",
                        "content": [
                            {
                                "type": "text",
                                "text": text,
                                "annotations": [
                                    {
                                        "type": "speech_metadata",
                                        "style": SPEECH_STYLE,
                                    }
                                ],
                            }
                        ],
                    }
                ],
                response_format={"type": "audio"},
                generation_config={
                    "speech_config": [{"voice": VOICE}],
                },
            )
            break
        except Exception as exc:
            status_code = retryable_status_code(exc)
            can_retry = status_code in {429, 500, 502, 503, 504}
            if attempt == MAX_RETRIES or not can_retry:
                raise
            wait_seconds = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            print(
                f"Gemini lỗi tạm thời {status_code}; thử lại sau {wait_seconds}s "
                f"({attempt}/{MAX_RETRIES})."
            )
            time.sleep(wait_seconds)

    output_audio = getattr(interaction, "output_audio", None)
    encoded_audio = getattr(output_audio, "data", None)
    if not encoded_audio:
        raise RuntimeError("Gemini không trả về output_audio.data.")

    try:
        audio = base64.b64decode(encoded_audio, validate=True)
    except (ValueError, TypeError) as exc:
        raise RuntimeError("Gemini trả về audio Base64 không hợp lệ.") from exc
    if not audio:
        raise RuntimeError("Gemini trả về audio rỗng.")

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / (
        f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_gemini_customer.wav"
    )
    output_path.write_bytes(audio)

    metadata = {
        "interaction_id": getattr(interaction, "id", None),
        "mime_type": getattr(output_audio, "mime_type", None) or "audio/wav",
        "text_characters": len(text),
        "model": MODEL,
        "voice": VOICE,
    }
    return output_path, len(audio), metadata

## 4. Gọi Gemini và lưu WAV

Ô này chỉ gọi API khi `RUN_TTS = True`. Kết quả được lưu trong `/content/<scenario_id>/`, kèm `manifest_gemini.json`.

In [ ]:
if not RUN_TTS:
    print("Chưa gọi API. Hãy đổi RUN_TTS = True trong ô cấu hình khi sẵn sàng.")
else:
    if not GEMINI_API_KEY:
        raise EnvironmentError(
            "Thiếu GEMINI_API_KEY trong Colab Secrets, /content/.env hoặc biến môi trường."
        )

    if CALL_IDS is None:
        selected_call_ids = list(customer_turns_by_call)
    else:
        if isinstance(CALL_IDS, str):
            raise TypeError("CALL_IDS phải là list, ví dụ ['call_1'], không phải chuỗi.")
        selected_call_ids = list(dict.fromkeys(CALL_IDS))
        missing = [call_id for call_id in selected_call_ids if call_id not in customer_turns_by_call]
        if missing:
            raise KeyError(f"CALL_IDS không tồn tại trong file: {missing}")

    scenario_output_dir = OUTPUT_DIR / safe_filename(scenario_id)
    results = []
    generated_files = []
    client = genai.Client(api_key=GEMINI_API_KEY)

    try:
        for call_id in selected_call_ids:
            expected_path = scenario_output_dir / (
                f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_gemini_customer.wav"
            )
            if expected_path.exists() and not OVERWRITE:
                print(f"Bỏ qua file đã tồn tại: {expected_path.name}")
                results.append(
                    {
                        "call_id": call_id,
                        "status": "skipped_existing",
                        "output_file": str(expected_path),
                    }
                )
                continue

            try:
                output_path, byte_count, metadata = synthesize_call(
                    client=client,
                    scenario_id=scenario_id,
                    call_id=call_id,
                    text_chunks=customer_turns_by_call[call_id],
                    output_dir=scenario_output_dir,
                )
                generated_files.append(output_path)
                results.append(
                    {
                        "call_id": call_id,
                        "status": "generated",
                        "turn_count": len(customer_turns_by_call[call_id]),
                        "bytes": byte_count,
                        "output_file": str(output_path),
                        **metadata,
                    }
                )
                print(f"Đã ghi {byte_count:,} bytes: {output_path}")
            except Exception as exc:
                results.append(
                    {
                        "call_id": call_id,
                        "status": "error",
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )
                print(f"Lỗi ở {call_id}: {type(exc).__name__}: {exc}")
    finally:
        close_client = getattr(client, "close", None)
        if callable(close_client):
            close_client()

    scenario_output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = scenario_output_dir / "manifest_gemini.json"
    manifest = {
        "source_file": str(SCENARIO_FILE),
        "scenario_id": scenario_id,
        "model": MODEL,
        "voice": VOICE,
        "speech_style": SPEECH_STYLE,
        "results": results,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)

    if generated_files:
        print("Nghe thử file vừa tạo đầu tiên:")
        display(Audio(filename=str(generated_files[0])))